# M2.1 · Target leakage

_Curriculum · Domain 0 · ML Foundations · Feature engineering & leakage_

**A feature that secretly encodes the label makes offline scores soar and production scores collapse.**

We reproduce leakage on purpose: add a feature built from the label, watch validation AUC (area under the ROC curve) jump to near-perfect, then remove it and see the honest score. _Save a copy to your Drive (File -> Save a copy in Drive) to keep your edits._

In [ ]:
# Setup - numpy / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)

## First, build an honest dataset

Each row is one impression: three real features drive a rare click (about 6 percent positive). This is the data a correct pipeline would use.

In [ ]:
n = 4000
X = rng.normal(size=(n, 3))

true_w = np.array([1.1, -0.7, 0.4])
logits = X @ true_w - 3.0
p_true = 1.0 / (1.0 + np.exp(-logits))

y = (rng.random(n) < p_true).astype(int)

print("positive rate:", round(float(y.mean()), 4))

## The leak, in one line of math

A leaky feature is any coordinate of $x_t$ that depends on the future label $y_{t+\Delta}$. Here we inject the worst case: a near-copy of the label,

$$x_{\text{leak}} = y + \varepsilon,\qquad \varepsilon \sim \mathcal{N}(0,\, 0.01).$$

In [ ]:
# A post-outcome column: essentially the label with a little noise.
leak = y + rng.normal(scale=0.1, size=n)

X_leaky = np.column_stack([X, leak])

print("leaky design matrix shape:", X_leaky.shape)

### Train both models on the same split

We fit an honest model on the three real features and a leaky model that also sees `leak`, then compare validation AUC.

In [ ]:
Xtr, Xva, ytr, yva = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

Xtr_l, Xva_l = train_test_split(
    X_leaky, test_size=0.3, random_state=0, stratify=y
)[:2]

honest = LogisticRegression().fit(Xtr, ytr)
leaky = LogisticRegression().fit(Xtr_l, ytr)

auc_honest = roc_auc_score(yva, honest.predict_proba(Xva)[:, 1])
auc_leaky = roc_auc_score(yva, leaky.predict_proba(Xva_l)[:, 1])

print("honest val AUC:", round(auc_honest, 3))
print("leaky  val AUC:", round(auc_leaky, 3))

### The leak inflates the score

The leaky model looks far better offline only because it is reading the label off its own input. We assert the inflation is large - that gap is the fingerprint of leakage.

In [ ]:
gap = auc_leaky - auc_honest

# Leakage produces an implausible offline lift that will not survive serving.
assert auc_leaky > 0.95
assert gap > 0.15

print("leakage inflated AUC by:", round(gap, 3))

## Visualize the inflated score

The bars make the trap obvious: the leaky model's offline AUC is near-perfect, but only the honest bar reflects what production will actually see.

In [ ]:
labels = ["honest", "leaky"]
values = [auc_honest, auc_leaky]

fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(labels, values, color=["#4c78a8", "#e45756"])
ax.set_ylim(0.5, 1.0)
ax.set_ylabel("validation AUC")
ax.set_title("leakage inflates the offline score")
plt.show()

## Practice

Try each in the empty cell below.

1. Reduce the leak strength (raise the noise scale on `leak`) and watch the leaky AUC fall toward the honest one - leakage is a spectrum.
2. Replace the label-copy leak with a post-window count (e.g. `y * rng.poisson(3, n)`) and confirm it still leaks.
3. Remove the leaky column and re-verify that the honest AUC matches production expectations (around 0.7 here).

In [ ]:
# Your turn:
